# 🌍 Prédiction du Taux de Change EUR/TND
## Notebook 1 — Analyse Exploratoire des Données (EDA)
---
**Auteur :** *[Votre Nom]*  
**Date :** *2025*  
**Environnement :** Google Colab  


---
## 1. 🎯 Définition du Problème
---
### Contexte
Le taux de change **EUR/TND** (Euro / Dinar Tunisien) est une variable macroéconomique critique pour la Tunisie. Sa prévision précise est essentielle pour :
- Les entreprises importatrices/exportatrices
- Les décideurs de la Banque Centrale de Tunisie (BCT)
- Les investisseurs et acteurs des marchés financiers

### Objectif Technique
> **Prédire le taux de change EUR/TND quotidien** à partir d'indicateurs macroéconomiques tunisiens et internationaux, en utilisant des modèles de Machine Learning supervisé (régression).

### Variables utilisées
| Variable | Description | Source |
|---|---|---|
| `EURTND` | Taux de change EUR/TND (cible) | Données de marché |
| `PIB` | Produit Intérieur Brut (Tunisie) | Données économiques |
| `inflation` | Taux d'inflation (Tunisie) | BCT / INS |
| `taux_interet` | Taux directeur (Tunisie) | BCT |
| `Balance commerciale` | Solde commercial (Tunisie) | INS |
| `BRENT` | Prix du pétrole Brent (USD/baril) | Marchés internationaux |
| `USDEUR` | Taux USD/EUR | Marchés internationaux |

### Métriques d'évaluation prévues
- **R²** (coefficient de détermination)
- **RMSE** (Root Mean Squared Error)
- **MSE** (Mean Squared Error)


---
## 2. 📦 Importation des Librairies
---


In [ ]:
# Installation des librairies supplémentaires si nécessaire
!pip install xgboost lightgbm --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# Configuration globale des graphiques
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Librairies importées avec succès')

---
## 3. 📥 Chargement et Fusion des Données
---
Les données sont réparties dans plusieurs fichiers Excel (une variable par fichier). On fusionne tous les fichiers sur la colonne `Date` avec un **merge outer** pour conserver toutes les dates.


In [ ]:
# Chargement du fichier PIB journalisé (base du merge)
daily_df = pd.read_excel('/content/PIB_daily.xlsx')
daily_df['Date'] = pd.to_datetime(daily_df['Date'])

print(f'✅ PIB_daily chargé : {daily_df.shape[0]} lignes, {daily_df.shape[1]} colonnes')
display(daily_df.head())

In [ ]:
# Liste des fichiers à fusionner
excel_files = [
    '/content/taux_interet_quotidienne.xlsx',
    '/content/EURTND_close_prices.xlsx',
    '/content/final_balance_commerciale_daily.xlsx',
    '/content/Brent_close_prices.xlsx',
    '/content/USDEUR_close_prices.xlsx',
    '/content/final_inflation_daily.xlsx',
]

merged_df = daily_df.copy()

for file_path in excel_files:
    try:
        df_temp = pd.read_excel(file_path)

        # Standardisation du nom de la colonne date
        if 'Date' in df_temp.columns:
            df_temp['Date'] = pd.to_datetime(df_temp['Date'])
        elif 'date' in df_temp.columns:
            df_temp['Date'] = pd.to_datetime(df_temp['date'])
            df_temp = df_temp.drop(columns=['date'])
        else:
            df_temp.rename(columns={df_temp.columns[0]: 'Date'}, inplace=True)
            df_temp['Date'] = pd.to_datetime(df_temp['Date'])

        merged_df = pd.merge(merged_df, df_temp, on='Date', how='outer')
        print(f'  ✅ Fusionné : {os.path.basename(file_path)}')

    except Exception as e:
        print(f'  ❌ Erreur : {os.path.basename(file_path)} → {e}')

# Tri chronologique
merged_df = merged_df.sort_values(by='Date').reset_index(drop=True)

# Renommage des colonnes ambiguës
merged_df = merged_df.rename(columns={
    'close_x': 'EURTND',
    'close_y': 'BRENT',
    'close':   'USDEUR'
})

print(f'\n✅ Dataset final : {merged_df.shape[0]} lignes × {merged_df.shape[1]} colonnes')

---
## 4. 🔍 Aperçu Général du Dataset
---


In [ ]:
# Premières et dernières lignes
print('=== APERÇU DU DATASET ===')
display(merged_df.head())
display(merged_df.tail())

# Dimensions
print(f'\n📐 Dimensions : {merged_df.shape[0]} lignes × {merged_df.shape[1]} colonnes')
print(f'📅 Période couverte : {merged_df["Date"].min().date()} → {merged_df["Date"].max().date()}')

In [ ]:
# Types de données
print('=== TYPES DE DONNÉES ===')
print(merged_df.dtypes)

---
## 5. 📊 Statistiques Descriptives
---


In [ ]:
# Statistiques descriptives des variables numériques
desc = merged_df.drop(columns=['Date']).describe().T
desc['cv (%)'] = (desc['std'] / desc['mean'].abs() * 100).round(2)  # Coefficient de variation

print('=== STATISTIQUES DESCRIPTIVES ===')
display(desc.style.background_gradient(cmap='Blues', subset=['mean', 'std']))

---
## 6. ❓ Analyse des Valeurs Manquantes
---


In [ ]:
# Calcul des valeurs manquantes
missing = merged_df.isnull().sum()
missing_pct = (missing / len(merged_df) * 100).round(2)
missing_df = pd.DataFrame({'Valeurs manquantes': missing, 'Pourcentage (%)': missing_pct})
missing_df = missing_df[missing_df['Valeurs manquantes'] > 0].sort_values('Pourcentage (%)', ascending=False)

if missing_df.empty:
    print('✅ Aucune valeur manquante dans le dataset.')
else:
    print(f'⚠️ {len(missing_df)} colonnes contiennent des valeurs manquantes :')
    display(missing_df)

    # Heatmap des valeurs manquantes
    plt.figure(figsize=(14, 4))
    sns.heatmap(merged_df.drop(columns=['Date']).isnull(), cbar=False, cmap='Reds', yticklabels=False)
    plt.title('Heatmap des Valeurs Manquantes')
    plt.tight_layout()
    plt.show()

---
## 7. 📈 Visualisation des Séries Temporelles
---
On visualise l'évolution de chaque variable dans le temps pour détecter des tendances, des ruptures structurelles ou des anomalies.


In [ ]:
# Variables cibles et explicatives à visualiser
cols_to_plot = ['EURTND', 'BRENT', 'USDEUR', 'PIB', 'inflation', 'taux_interet', 'Balance commerciale']
cols_to_plot = [c for c in cols_to_plot if c in merged_df.columns]

fig, axes = plt.subplots(len(cols_to_plot), 1, figsize=(14, 3 * len(cols_to_plot)))

for i, col in enumerate(cols_to_plot):
    axes[i].plot(merged_df['Date'], merged_df[col], linewidth=1, color='steelblue')
    axes[i].set_title(f'Évolution de {col}', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Séries Temporelles des Variables Macroéconomiques', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 8. 📉 Distributions des Variables
---


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    axes[i].hist(merged_df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution de {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fréquence')

# Masquer les axes vides
for j in range(len(cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions des Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. 📦 Détection des Outliers (Boxplots)
---


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    axes[i].boxplot(merged_df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='steelblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'Outliers : {col}')
    axes[i].set_xticks([])

for j in range(len(cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Détection des Outliers (Boxplots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. 🔗 Analyse des Corrélations
---
La matrice de corrélation permet d'identifier les variables les plus liées à la cible **EUR/TND** et de détecter d'éventuelles redondances (multicolinéarité).


In [ ]:
# Matrice de corrélation (variables principales uniquement)
corr_cols = ['EURTND', 'BRENT', 'USDEUR', 'PIB', 'inflation', 'taux_interet', 'Balance commerciale']
corr_cols = [c for c in corr_cols if c in merged_df.columns]

corr_matrix = merged_df[corr_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('Matrice de Corrélation des Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec la cible EURTND (triées)
if 'EURTND' in corr_matrix.columns:
    corr_target = corr_matrix['EURTND'].drop('EURTND').sort_values(ascending=False)

    plt.figure(figsize=(8, 4))
    colors = ['green' if v > 0 else 'red' for v in corr_target]
    corr_target.plot(kind='bar', color=colors, edgecolor='white')
    plt.title('Corrélation de chaque variable avec EUR/TND', fontweight='bold')
    plt.ylabel('Coefficient de Pearson')
    plt.axhline(0, color='black', linewidth=0.8)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print('\n📌 Corrélations avec EURTND :')
    print(corr_target.to_string())

---
## 11. 🔄 Autocorrélation de la Variable Cible (EUR/TND)
---
L'autocorrélation mesure à quel point la valeur actuelle est corrélée avec ses valeurs passées (lags). Cela justifie l'utilisation de **variables de lag** dans la modélisation.


In [ ]:
from pandas.plotting import autocorrelation_plot

# Autocorrélation jusqu'à 30 lags
eurtnd_series = merged_df['EURTND'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Autocorrélation simple
lags = range(1, 31)
acf_values = [eurtnd_series.autocorr(lag=lag) for lag in lags]

axes[0].bar(lags, acf_values, color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(1.96/np.sqrt(len(eurtnd_series)), color='red', linestyle='--', alpha=0.7, label='Seuil 95%')
axes[0].axhline(-1.96/np.sqrt(len(eurtnd_series)), color='red', linestyle='--', alpha=0.7)
axes[0].set_title('Autocorrélation EUR/TND (Lags 1-30)', fontweight='bold')
axes[0].set_xlabel('Lag (jours)')
axes[0].set_ylabel('Corrélation')
axes[0].legend()

# Série temporelle EUR/TND
axes[1].plot(merged_df['Date'], merged_df['EURTND'], color='darkorange', linewidth=1)
axes[1].set_title('Évolution EUR/TND dans le temps', fontweight='bold')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

print(f'📌 Autocorrélation max (lag 1) : {eurtnd_series.autocorr(1):.4f}')
print(f'📌 Autocorrélation lag 7 : {eurtnd_series.autocorr(7):.4f}')
print(f'📌 Autocorrélation lag 30 : {eurtnd_series.autocorr(30):.4f}')

---
## 12. 📝 Synthèse de l'EDA
---

### Observations clés

| Observation | Détail |
|---|---|
| **Valeurs manquantes** | Présentes (comblées par forward/backward fill dans le preprocessing) |
| **Outliers** | Quelques valeurs extrêmes sur BRENT et Balance commerciale |
| **Tendance EURTND** | Tendance haussière visible sur la période étudiée |
| **Autocorrélation** | Forte autocorrélation → justifie les features lag (1 à 30 jours) |
| **Corrélations** | Corrélations significatives avec USDEUR, inflation et BRENT |

### Décisions pour la modélisation
- ✅ Utilisation de **30 lags** de la cible comme features principales
- ✅ **Forward fill + Backward fill** pour les valeurs manquantes
- ✅ **Standardisation (StandardScaler)** avant la modélisation
- ✅ **Split temporel chronologique** (pas de split aléatoire) pour respecter la nature de la série
- ✅ Algorithmes testés : **XGBoost, [Modèle 2], [Modèle 3]**

> ➡️ La suite du pipeline est dans le notebook **`02_Modeling.ipynb`**
